# Building Your First Neural Network on the MNIST Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version :', tf.__version__)
print('GPU available       :', bool(tf.config.list_physical_devices('GPU')))

## 1. Load and Preprocess the MNIST Dataset

In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

print('=== Raw dataset shapes ===')
print(f'  X_train : {X_train_raw.shape}   ({X_train_raw.shape[0]:,} images)')
print(f'  X_test  : {X_test_raw.shape}    ({X_test_raw.shape[0]:,} images)')
print(f'  Pixel range (raw): {X_train_raw.min()} - {X_train_raw.max()}')

In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train_raw / 255.0
X_test  = X_test_raw  / 255.0
print(f'Pixel range after normalisation: {X_train.min():.1f} - {X_train.max():.1f}')

In [ ]:
# One-hot encode labels
NUM_CLASSES = 10
y_train_ohe = to_categorical(y_train_raw, NUM_CLASSES)
y_test_ohe  = to_categorical(y_test_raw,  NUM_CLASSES)

print(f'Label before OHE : {y_train_raw[0]}')
print(f'Label after  OHE : {y_train_ohe[0]}')

In [ ]:
# Display sample images
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
np.random.seed(42)
sample_idx = np.random.choice(len(X_train), 32, replace=False)

for ax, idx in zip(axes.flatten(), sample_idx):
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(str(y_train_raw[idx]), fontsize=12, fontweight='bold', color='steelblue')
    ax.axis('off')

plt.suptitle('Sample MNIST Images (label above each image)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y, title in [(axes[0], y_train_raw, 'Training Set'), (axes[1], y_test_raw, 'Test Set')]:
    counts = np.bincount(y)
    ax.bar(range(10), counts, color=plt.cm.tab10(np.linspace(0,1,10)), edgecolor='white')
    ax.set_title(f'Digit Distribution - {title}', fontweight='bold')
    ax.set_xlabel('Digit')
    ax.set_ylabel('Count')
    ax.set_xticks(range(10))
    ax.grid(axis='y', alpha=0.3)
    for i, c in enumerate(counts):
        ax.text(i, c+50, str(c), ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 2. Build the Fully Connected Neural Network

In [ ]:
def build_model(units_1=128, units_2=64, dropout=0.0, learning_rate=1e-3):
    """
    Fully-connected neural network for MNIST.
    Architecture: Flatten -> Dense(units_1, ReLU) -> Dense(units_2, ReLU) -> Dense(10, Softmax)
    """
    m = models.Sequential([
        layers.Flatten(input_shape=(28, 28)),
        layers.Dense(units_1, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(units_2, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='MNIST_NN')
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return m

model = build_model()
model.summary()

In [ ]:
print('Network Architecture')
print('-'*50)
print('  Input:   28 x 28 grayscale image')
print('  Flatten: 784-dimensional vector')
print('  Dense 1: 128 neurons | Activation: ReLU')
print('  Dense 2:  64 neurons | Activation: ReLU')
print('  Output:   10 neurons | Activation: Softmax')
print('-'*50)
print(f'  Total parameters: {model.count_params():,}')
print()
print('Compilation: Adam (lr=0.001), CategoricalCrossentropy, Accuracy')

## 3. Train the Neural Network

In [ ]:
EPOCHS     = 10
BATCH_SIZE = 128
VAL_SPLIT  = 0.10

history = model.fit(
    X_train, y_train_ohe,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = VAL_SPLIT,
    verbose          = 1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs_range = range(1, EPOCHS + 1)

# Accuracy
axes[0].plot(epochs_range, history.history['accuracy'],     'o-', color='#4C72B0', linewidth=2, markersize=5, label='Train')
axes[0].plot(epochs_range, history.history['val_accuracy'], 's-', color='#E84040', linewidth=2, markersize=5, label='Validation')
axes[0].set_title('Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].set_xticks(epochs_range)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(epochs_range, history.history['loss'],     'o-', color='#4C72B0', linewidth=2, markersize=5, label='Train')
axes[1].plot(epochs_range, history.history['val_loss'], 's-', color='#E84040', linewidth=2, markersize=5, label='Validation')
axes[1].set_title('Loss over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Categorical Cross-Entropy Loss')
axes[1].legend()
axes[1].set_xticks(epochs_range)
axes[1].grid(True, alpha=0.3)

plt.suptitle('MNIST Neural Network - Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Evaluate the Model's Performance

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
y_proba = model.predict(X_test, verbose=0)
y_pred  = y_proba.argmax(axis=1)
y_true  = y_test_raw

print(classification_report(y_true, y_pred, target_names=[str(i) for i in range(10)]))

In [ ]:
# Confusion matrix
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.5,
            xticklabels=range(10), yticklabels=range(10), ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Digit')
axes[0].set_ylabel('True Digit')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5,
            xticklabels=range(10), yticklabels=range(10), ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalised)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Digit')
axes[1].set_ylabel('True Digit')

plt.suptitle('Confusion Matrix - MNIST Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy
per_class_acc = cm_norm.diagonal()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E84040' if v < per_class_acc.mean() else '#55A868' for v in per_class_acc]
bars = ax.bar(range(10), per_class_acc * 100, color=colors, edgecolor='white')
ax.axhline(per_class_acc.mean() * 100, color='navy', linestyle='--', linewidth=1.5,
           label=f'Average ({per_class_acc.mean()*100:.1f}%)')
for bar, val in zip(bars, per_class_acc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val*100:.1f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_title('Per-Class Accuracy on Test Set', fontsize=13, fontweight='bold')
ax.set_xlabel('Digit')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(range(10))
ax.set_ylim(85, 105)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

hardest = per_class_acc.argmin()
easiest = per_class_acc.argmax()
print(f'Hardest digit : {hardest} ({per_class_acc[hardest]*100:.1f}% accuracy)')
print(f'Easiest digit : {easiest} ({per_class_acc[easiest]*100:.1f}% accuracy)')

In [ ]:
# Misclassified examples
errors = np.where(y_pred != y_true)[0]
print(f'Misclassified: {len(errors)} / {len(X_test):,} ({len(errors)/len(X_test)*100:.2f}%)')

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for ax, idx in zip(axes.flatten(), errors[:32]):
    ax.imshow(X_test[idx], cmap='gray')
    conf = y_proba[idx][y_pred[idx]] * 100
    ax.set_title(f'P:{y_pred[idx]} T:{y_true[idx]}\n{conf:.0f}%', fontsize=8, color='crimson', fontweight='bold')
    ax.axis('off')

plt.suptitle('Misclassified Images (P=Predicted, T=True)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning Experiment

In [ ]:
configs = [
    {'units_1':  64, 'units_2': 32,  'dropout': 0.0, 'learning_rate': 1e-3, 'label': '64-32, lr=1e-3'},
    {'units_1': 128, 'units_2': 64,  'dropout': 0.0, 'learning_rate': 1e-3, 'label': '128-64, lr=1e-3 (baseline)'},
    {'units_1': 256, 'units_2': 128, 'dropout': 0.0, 'learning_rate': 1e-3, 'label': '256-128, lr=1e-3'},
    {'units_1': 128, 'units_2': 64,  'dropout': 0.3, 'learning_rate': 1e-3, 'label': '128-64 + Dropout(0.3)'},
    {'units_1': 128, 'units_2': 64,  'dropout': 0.0, 'learning_rate': 1e-4, 'label': '128-64, lr=1e-4'},
]

TUNE_EPOCHS = 5
results_hp  = []

for cfg in configs:
    label = cfg.pop('label')
    m = build_model(**cfg)
    hist = m.fit(X_train, y_train_ohe, epochs=TUNE_EPOCHS, batch_size=128,
                 validation_split=0.1, verbose=0)
    _, tacc = m.evaluate(X_test, y_test_ohe, verbose=0)
    results_hp.append({'Config': label, 'Val Acc': max(hist.history['val_accuracy']),
                       'Test Acc': tacc, 'history': hist.history})
    cfg['label'] = label
    print(f'{label:<40}  Test Acc = {tacc*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
palette = plt.cm.tab10(np.linspace(0, 0.5, len(results_hp)))

for res, color in zip(results_hp, palette):
    axes[0].plot(range(1, TUNE_EPOCHS+1), res['history']['val_accuracy'],
                 marker='o', linewidth=2, markersize=5, color=color, label=res['Config'])
axes[0].set_title('Validation Accuracy - All Configurations', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].legend(fontsize=7, loc='lower right')
axes[0].grid(True, alpha=0.3)

labels_hp = [r['Config'] for r in results_hp]
test_vals  = [r['Test Acc']*100 for r in results_hp]
bars = axes[1].barh(labels_hp, test_vals, color=palette, edgecolor='white')
for bar, val in zip(bars, test_vals):
    axes[1].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                 f'{val:.2f}%', va='center', fontsize=9, fontweight='bold')
axes[1].set_title('Test Accuracy by Configuration', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Test Accuracy (%)')
axes[1].set_xlim(90, 100)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Hyperparameter Tuning Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best = max(results_hp, key=lambda x: x['Test Acc'])
print(f'Best configuration : {best["Config"]}')
print(f'Best test accuracy : {best["Test Acc"]*100:.2f}%')

In [ ]:
print('='*55)
print('  MNIST NEURAL NETWORK - SUMMARY')
print('='*55)
print(f'  Training samples  : {len(X_train):,}')
print(f'  Test samples      : {len(X_test):,}')
print()
print('  Baseline model architecture:')
print('    Flatten -> Dense(128,ReLU) -> Dense(64,ReLU) -> Dense(10,Softmax)')
print(f'  Baseline test accuracy : {test_acc*100:.2f}%')
print(f'  Misclassified images   : {len(errors):,}')
print()
print(f'  Hardest digit: {hardest} ({per_class_acc[hardest]*100:.1f}% accuracy)')
print(f'  Easiest digit: {easiest} ({per_class_acc[easiest]*100:.1f}% accuracy)')
print()
print(f'  Best hyperparameter config : {best["Config"]}')
print(f'  Best test accuracy         : {best["Test Acc"]*100:.2f}%')